# SHAP Analysis — JJUSHYCSH Referral No-Show Risk Model (BAN6800 Module 4)
**Abdi Farah Ahmed · Nexford University**

Global and local explanations for the champion model saved by `run_all.py`, implementing the Module 2 explainability framework on the **Module 3 feature definitions**. Run this notebook **after** `python run_all.py` (from the `module4` folder) and keep the executed outputs when you commit it to GitHub.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib, numpy as np, pandas as pd, shap, matplotlib.pyplot as plt
from m4 import config as C
from m4 import data as D
from m4.xai_fairness import pretty

bundle = joblib.load(ROOT / 'outputs' / 'model' / 'noshow_model.joblib')
pipe, t_high, FEATURES = bundle['pipeline'], bundle['threshold_high'], bundle['features']
print('Champion:', bundle['algorithm'], '| operating threshold:', round(t_high, 3))
print('Features:', FEATURES, '| dropped after necessity review:', bundle.get('dropped_flags'))

In [ ]:
# Module 3 hand-over: the same test split used in the report (data/processed/test.*)
parts, info = D.load_inputs(processed_dir=ROOT.parent / 'data' / 'processed')
test = D.apply_minimisation(parts['test'], bundle.get('dropped_flags') or [])
print(info['route'], '| test appointments:', len(test))
prep, clf = pipe.named_steps['prep'], pipe.named_steps['clf']
Xs = test[FEATURES].sample(min(2000, len(test)), random_state=C.RANDOM_STATE)
Xt = pd.DataFrame(prep.transform(Xs), columns=prep.get_feature_names_out(), index=Xs.index)
display_vals = Xt.copy()
for c in C.NUMERIC:
    if c in Xs.columns:
        display_vals[c] = Xs[c].values   # show days / years / counts, not z-scores
print('explaining a sample of', len(Xs))

## 1. Build the explainer
TreeExplainer for tree ensembles, LinearExplainer for logistic regression (Lundberg & Lee, 2017).

In [ ]:
if clf.__class__.__name__ == 'LogisticRegression':
    explainer = shap.LinearExplainer(clf, Xt)
else:
    explainer = shap.TreeExplainer(clf)
sv = explainer(Xt)
vals, bv = np.asarray(sv.values, float), np.asarray(sv.base_values, float)
if vals.ndim == 3:
    vals = vals[:, :, 1]; bv = bv[..., 1] if bv.ndim >= 1 and bv.shape[-1] == 2 else bv
bv = np.full(len(vals), float(bv.ravel()[0])) if bv.size == 1 else bv.reshape(len(vals), -1)[:, -1]
exp = shap.Explanation(values=vals, base_values=bv, data=display_vals.values.astype(float),
                       feature_names=[pretty(f) for f in Xt.columns])
print(type(explainer).__name__)

## 2. Global importance
Which features drive predicted no-show risk across appointments?

In [ ]:
shap.plots.beeswarm(exp, max_display=12)

In [ ]:
shap.plots.bar(exp, max_display=12)
imp = pd.Series(np.abs(exp.values).mean(0), index=exp.feature_names).sort_values(ascending=False)
imp.head(10).round(4)

## 3. Dependence: how lead time and attendance history shape risk

In [ ]:
shap.plots.scatter(exp[:, 'Lead time (days booked ahead)'], color=exp[:, 'Patient age'])

In [ ]:
shap.plots.scatter(exp[:, 'Prior no-show rate'], color=exp[:, 'Number of prior visits'])

## 4. Local explanations for individual appointments
The reason codes the API returns to OPD coordinators come from these same contributions (Module 2: audience-specific explainability).

In [ ]:
p = pipe.predict_proba(Xs)[:, 1]
i_hi, i_lo = int(np.argmax(p)), int(np.argmin(p))
print(f'High-risk example p={p[i_hi]:.2f}')
shap.plots.waterfall(exp[i_hi], max_display=10)

In [ ]:
print(f'Low-risk example p={p[i_lo]:.2f}')
shap.plots.waterfall(exp[i_lo], max_display=10)

In [ ]:
# A borderline case near the operating threshold - the kind a coordinator would query
i_mid = int(np.argmin(np.abs(p - t_high)))
print(f'Borderline example p={p[i_mid]:.2f} (threshold {t_high:.2f})')
shap.plots.waterfall(exp[i_mid], max_display=10)

## 5. Proxy check: does the score track protected attributes it never saw?
Gender and area_code are not model inputs; if scores differ strongly by them, another feature is acting as a proxy.

In [ ]:
chk = test.loc[Xs.index].assign(score=p)
print(chk.groupby('gender')['score'].mean().round(3))
print(chk.groupby('age_band')['score'].mean().round(3))
print('Spread of mean score across area codes:', round(chk.groupby('area_code')['score'].mean().std(), 3))

## 6. Interpretation (write in your own words)

- **Top drivers and what they mean for JJUSHYCSH outreach:** …
- **Does anything look like a proxy for a protected attribute (gender, area, disability)?** …
- **What would you tell an OPD coordinator about the borderline case?** …